# 03 — Traces: collection & visualization

Pull traces from Tempo (or any OTLP-compatible store with the Tempo HTTP API), turn them into a span DataFrame, and chart latency / call structure.

What you'll do:

1. Search Tempo for recent traces matching a TraceQL filter.
2. Fetch a single trace and flatten it into a span table.
3. Compute per-service latency percentiles.
4. Visualize one trace as a Gantt-style waterfall.

In [ ]:
import os

TEMPO_URL = os.environ.get('TEMPO_URL', 'http://tempo:3200')
TEMPO_TENANT = os.environ.get('TEMPO_TENANT_ID', '')
TRACEQL = '{ duration > 100ms }'
RANGE_HOURS = 1
LIMIT = 20

In [ ]:
import time, requests, pandas as pd

def _headers():
    return {'X-Scope-OrgID': TEMPO_TENANT} if TEMPO_TENANT else {}

def search_traces(traceql, hours=RANGE_HOURS, limit=LIMIT):
    end = int(time.time())
    start = end - hours * 3600
    r = requests.get(f'{TEMPO_URL}/api/search',
                     params={'q': traceql, 'start': start, 'end': end, 'limit': limit},
                     headers=_headers(), timeout=60)
    r.raise_for_status()
    return r.json().get('traces', [])

def fetch_trace(trace_id):
    r = requests.get(f'{TEMPO_URL}/api/traces/{trace_id}', headers=_headers(), timeout=60)
    r.raise_for_status()
    return r.json()

traces = search_traces(TRACEQL)
print(f'found {len(traces)} traces')
pd.DataFrame(traces)[['traceID', 'rootServiceName', 'rootTraceName', 'durationMs']].head(10) if traces else 'no traces'

## Flatten a trace into a span DataFrame

Tempo returns OTLP JSON: `{batches: [{resource, scopeSpans: [{spans: [...]}]}]}`. Each span carries `startTimeUnixNano`, `endTimeUnixNano`, `name`, `kind`, attributes.

In [ ]:
def attrs_to_dict(attrs):
    out = {}
    for a in attrs or []:
        v = a.get('value', {})
        out[a['key']] = next(iter(v.values())) if v else None
    return out

def trace_to_spans(trace):
    rows = []
    for batch in trace.get('batches', []):
        res = attrs_to_dict(batch.get('resource', {}).get('attributes', []))
        svc = res.get('service.name', '?')
        for scope in batch.get('scopeSpans', []):
            for s in scope.get('spans', []):
                start_ns = int(s['startTimeUnixNano'])
                end_ns = int(s['endTimeUnixNano'])
                rows.append({
                    'service': svc,
                    'name': s['name'],
                    'span_id': s['spanId'],
                    'parent_id': s.get('parentSpanId') or None,
                    'kind': s.get('kind'),
                    'start': pd.to_datetime(start_ns, unit='ns'),
                    'end': pd.to_datetime(end_ns, unit='ns'),
                    'duration_ms': (end_ns - start_ns) / 1e6,
                })
    return pd.DataFrame(rows)

spans = pd.DataFrame()
if traces:
    t = fetch_trace(traces[0]['traceID'])
    spans = trace_to_spans(t)
spans.head()

## Latency percentiles by service (sampled set)

In [ ]:
all_spans = pd.DataFrame()
for tr in traces[:10]:
    all_spans = pd.concat([all_spans, trace_to_spans(fetch_trace(tr['traceID']))], ignore_index=True)

if not all_spans.empty:
    all_spans.groupby('service')['duration_ms'].describe(percentiles=[0.5, 0.9, 0.99]).round(1)

## Waterfall view of a single trace

In [ ]:
import matplotlib.pyplot as plt

if not spans.empty:
    s = spans.sort_values('start').reset_index(drop=True)
    t0 = s['start'].min()
    s['offset_ms'] = (s['start'] - t0).dt.total_seconds() * 1000
    fig, ax = plt.subplots(figsize=(12, max(3, len(s) * 0.25)))
    ax.barh(range(len(s)), s['duration_ms'], left=s['offset_ms'])
    ax.set_yticks(range(len(s)))
    ax.set_yticklabels([f"{r.service}: {r['name']}" for _, r in s.iterrows()], fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel('ms from trace start')
    ax.set_title(f"Trace {traces[0]['traceID']}")
    plt.tight_layout()

### Where to go next

- Define a service-graph SLO with Pyrra/Sloth using the same span data.
- Feed `traceQLMetrics` into a Grafana dashboard rendered by this workspace.